In [ ]:
#create yearly detections
import os
import arcpy

arcpy.env.overwriteOutput = True

# Define base directory and working GDB 
base_dir = os.path.dirname(os.path.abspath("__file__"))
working_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")

master = f"{gdb}\\merged_94_24_nlcd_size_filtered"

arcpy.env.overwriteOutput = True

for year in range(1994, 2025):

    out_fc = f"{gdb}\\SEFM_raw_{year}"

    if arcpy.Exists(out_fc):
        arcpy.management.Delete(out_fc)

    where = f"year = {year}"

    arcpy.analysis.Select(
        in_features=master,
        out_feature_class=out_fc,
        where_clause=where
    )

    print(f"Created {out_fc}")

In [ ]:
#function for spatial/temporal clustering
def assign_events_for_year(year):

    import arcpy
    from datetime import datetime

    # ------------------------------------------------------------
    # Setup
    # ------------------------------------------------------------
    gdb = working_gdb
    fc = f"{gdb}\\SEFM_raw_{year}"              # raw polygons for this year
    buffer_fc = f"{gdb}\\SEFM_90m_buffer_{year}" # 90 m buffer
    sj_fc = f"{gdb}\\sj_{year}"                  # spatial join output

    arcpy.env.overwriteOutput = True

    # Ensure event_id field exists
    fields = [f.name for f in arcpy.ListFields(fc)]
    if "event_id" not in fields:
        arcpy.management.AddField(fc, "event_id", "TEXT", field_length=20)

    # ------------------------------------------------------------
    # 1. Create 90 m buffer for adjacency
    # ------------------------------------------------------------
    arcpy.analysis.Buffer(
        in_features=fc,
        out_feature_class=buffer_fc,
        buffer_distance_or_field="90 Meters"
    )

    # ------------------------------------------------------------
    # 2. Spatial join: buffer → raw polygons
    # ------------------------------------------------------------
    arcpy.analysis.SpatialJoin(
        target_features=buffer_fc,
        join_features=fc,
        out_feature_class=sj_fc,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    # ------------------------------------------------------------
    # 3. Build adjacency list using detection_id
    # ------------------------------------------------------------
    adj = {}
    with arcpy.da.SearchCursor(sj_fc, ["detection_id", "detection_id_1"]) as cur:
        for tid, jid in cur:
            if tid == jid:
                continue
            adj.setdefault(tid, set()).add(jid)
            adj.setdefault(jid, set()).add(tid)

    # Ensure all detections appear in adjacency
    all_ids = [row[0] for row in arcpy.da.SearchCursor(fc, ["detection_id"])]
    for det in all_ids:
        adj.setdefault(det, set())

    # ------------------------------------------------------------
    # 4. Build spatial clusters (connected components)
    # ------------------------------------------------------------
    visited = set()
    spatial_clusters = []

    for det in all_ids:
        if det in visited:
            continue
        stack = [det]
        cluster = []
        while stack:
            x = stack.pop()
            if x in visited:
                continue
            visited.add(x)
            cluster.append(x)
            stack.extend(adj[x] - visited)
        spatial_clusters.append(cluster)

    # ------------------------------------------------------------
    # 5. Load temporal intervals
    # ------------------------------------------------------------
    intervals = {}
    with arcpy.da.SearchCursor(
        fc,
        ["detection_id", "prebd_min_corrected", "bd_min_corrected_plus8"]
    ) as cur:
        for det, start, end in cur:
            intervals[det] = (start, end)

    # ------------------------------------------------------------
    # 6. Temporal grouping within each spatial cluster
    # ------------------------------------------------------------
    event_lookup = {}
    multi_counter = 1
    single_counter = 1

    for cluster in spatial_clusters:

        # Single polygon cluster → singleton event
        if len(cluster) == 1:
            det = cluster[0]
            event_lookup[det] = f"S{year}_{single_counter:04d}"
            single_counter += 1
            continue

        # Sort by start time
        cluster_sorted = sorted(cluster, key=lambda x: intervals[x][0])

        current_group = []
        groups = []

        for det in cluster_sorted:
            start, end = intervals[det]

            if not current_group:
                current_group = [det]
                last_end = end
                continue

            if start <= last_end:
                current_group.append(det)
                last_end = max(last_end, end)
            else:
                groups.append(current_group)
                current_group = [det]
                last_end = end

        if current_group:
            groups.append(current_group)

        # Assign event IDs to temporal groups
        for g in groups:
            if len(g) == 1:
                det = g[0]
                event_lookup[det] = f"S{year}_{single_counter:04d}"
                single_counter += 1
            else:
                eid = f"{year}_{multi_counter:04d}"
                for det in g:
                    event_lookup[det] = eid
                multi_counter += 1

    # ------------------------------------------------------------
    # 7. Write event_ids back to the feature class
    # ------------------------------------------------------------
    with arcpy.da.UpdateCursor(fc, ["detection_id", "event_id"]) as cur:
        for row in cur:
            det = row[0]
            row[1] = event_lookup[det]
            cur.updateRow(row)

In [ ]:
#test one year
assign_events_for_year(2020)

In [ ]:
#seems ok, try all years
for year in range(1994, 2025):
    assign_events_for_year(year)

In [ ]:
#join event_id back to larger dataset
import arcpy

master = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")

# 1. Ensure event_id exists on master
fields = [f.name for f in arcpy.ListFields(master)]
if "event_id" not in fields:
    arcpy.management.AddField(master, "event_id", "TEXT", field_length=20)

# 2. Build global lookup: detection_id -> event_id
lookup = {}

for year in range(1994, 2025):
    raw_fc = f"{gdb}\\SEFM_raw_{year}"
    with arcpy.da.SearchCursor(raw_fc, ["detection_id", "event_id"]) as cur:
        for det, eid in cur:
            if det is not None and eid is not None:
                lookup[det] = eid

print(f"Lookup size: {len(lookup)} detection_ids")

# 3. Update master using the lookup
with arcpy.da.UpdateCursor(master, ["detection_id", "event_id"]) as cur:
    for det, eid in cur:
        if det in lookup:
            cur.updateRow([det, lookup[det]])

In [ ]:
# Calculate duration of the detection window
import arcpy
from datetime import datetime

master = f"{gdb}\\merged_94_24_nlcd_size_filtered"

arcpy.env.overwriteOutput = True

# 1. Ensure window_days field exists
fields = [f.name for f in arcpy.ListFields(master)]
if "window_days" not in fields:
    arcpy.management.AddField(master, "window_days", "LONG")

# 2. Calculate the difference in days
with arcpy.da.UpdateCursor(master, ["prebd_min_corrected", "bd_min_corrected_plus8", "window_days"]) as cur:
    for pre_bd, bd_plus8, win_days in cur:
        
        # Guard against any edge case nulls or zeros
        if None in (pre_bd, bd_plus8) or 0 in (pre_bd, bd_plus8):
            continue
            
        # Parse start date
        s_start = str(int(pre_bd))
        start_date = datetime(int(s_start[0:4]), int(s_start[4:6]), int(s_start[6:8]))
        
        # Parse end date
        s_end = str(int(bd_plus8))
        end_date = datetime(int(s_end[0:4]), int(s_end[4:6]), int(s_end[6:8]))
        
        # Subtract dates to get timedelta object and extract the days
        delta = end_date - start_date
        
        # Update row with the absolute number of days
        cur.updateRow([pre_bd, bd_plus8, abs(delta.days)])

print("Successfully calculated window_days duration field.")

In [ ]:
#count detections per event
import arcpy
from collections import Counter

master = f"{gdb}\\merged_94_24_nlcd_size_filtered"

event_sizes = Counter()

with arcpy.da.SearchCursor(master, ["event_id"]) as cur:
    for (eid,) in cur:
        if eid is not None:
            event_sizes[eid] += 1

In [ ]:
#count single vs multi detection events
single_events = sum(1 for size in event_sizes.values() if size == 1)
multi_events  = sum(1 for size in event_sizes.values() if size > 1)

total_events = single_events + multi_events

In [ ]:
#calculate percent multi vs single events
pct_single = 100 * single_events / total_events
pct_multi  = 100 * multi_events / total_events

print("Single‑detection events:", pct_single)
print("Multi‑detection events:", pct_multi)

In [ ]:
#range
min_size = min(multi_sizes)
max_size = max(multi_sizes)

print("Minimum detections in a multi‑detection event:", min_size)
print("Maximum detections in a multi‑detection event:", max_size)

In [ ]:
# dissolve by event_id and calculate mode for year, ecoregion, and land cover
import arcpy
from collections import Counter

in_fc = os.path.join(working_gdb, "merged_94_24_nlcd_size_filtered")
out_fc = os.path.join(working_gdb, "SEFM_events_94_24")

arcpy.env.overwriteOutput = True

# 1. Run the Dissolve with placeholders for your mode fields
print("Running event-level dissolve...")
arcpy.management.Dissolve(
    in_features=in_fc,
    out_feature_class=out_fc,
    dissolve_field=["event_id"],
    statistics_fields=[
        ["prebd_min_corrected", "MIN"],
        ["bd_min_corrected_plus8", "MAX"],
        ["bd_min_corrected", "MIN"],
        ["area_ha", "SUM"],
        ["detection_id", "COUNT"],
        ["year", "FIRST"],         # Placeholder for year_mode
        ["ecol3", "FIRST"],        # Placeholder for ecol3_mode
        ["nlcdr_domi", "FIRST"]    # Placeholder for nlcdr_mode
    ],
    multi_part="MULTI_PART"
)

print("Dissolve complete. Building in-memory frequency dictionaries...")

# 2. Initialize dictionaries to track all values per event
event_year_counts = {}
event_ecol3_counts = {}
event_nlcdr_counts = {}

# Read the raw data to capture all entries associated with each event
fields_to_read = ["event_id", "year", "ecol3", "nlcdr_domi"]
with arcpy.da.SearchCursor(in_fc, fields_to_read) as cur:
    for eid, yr, eco, nlc in cur:
        if eid is not None:
            if yr is not None:
                event_year_counts.setdefault(eid, []).append(yr)
            if eco is not None:
                event_ecol3_counts.setdefault(eid, []).append(eco)
            if nlc is not None:
                event_nlcdr_counts.setdefault(eid, []).append(nlc)

# Extract the single most common value (mode) for each unique event
event_mode_year = {eid: Counter(yrs).most_common(1)[0][0] for eid, yrs in event_year_counts.items()}
event_mode_ecol3 = {eid: Counter(ecos).most_common(1)[0][0] for eid, ecos in event_ecol3_counts.items()}
event_mode_nlcdr = {eid: Counter(nlcs).most_common(1)[0][0] for eid, nlcs in event_nlcdr_counts.items()}

print("Writing modes back to the dissolved event feature class...")

# 3. Overwrite the default "FIRST" columns with calculated modes
fields_to_update = ["event_id", "FIRST_year", "FIRST_ecol3", "FIRST_nlcdr_domi"]
with arcpy.da.UpdateCursor(out_fc, fields_to_update) as cur:
    for row in cur:
        eid = row[0]
        if eid in event_mode_year:
            row[1] = event_mode_year[eid]
        if eid in event_mode_ecol3:
            row[2] = event_mode_ecol3[eid]
        if eid in event_mode_nlcdr:
            row[3] = event_mode_nlcdr[eid]
        cur.updateRow(row)

# 4. Clean up the default field names to your preferred format
print("Renaming fields to custom mode format...")
arcpy.management.AlterField(out_fc, "FIRST_year", "year_mode", "year_mode")
arcpy.management.AlterField(out_fc, "FIRST_ecol3", "ecol3_mode", "ecol3_mode")
arcpy.management.AlterField(out_fc, "FIRST_nlcdr_domi", "nlcdr_mode", "nlcdr_mode")

print("All modes successfully calculated, applied, and fields renamed.")

In [ ]:
# Calculate total event window duration in days
import arcpy
from datetime import datetime

out_fc = os.path.join(working_gdb, "SEFM_events_94_24")

# 1. Ensure event_window_days field exists
existing_fields = [f.name for f in arcpy.ListFields(out_fc)]
if "event_window_days" not in existing_fields:
    arcpy.management.AddField(out_fc, "event_window_days", "LONG", field_alias="event_window_days")

print("Calculating event_window_days...")

# 2. Map out the field names generated by the Dissolve tool
fields = ["MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", "event_window_days"]

with arcpy.da.UpdateCursor(out_fc, fields) as cur:
    for min_prebd, max_plus8, win_days in cur:
        
        # Guard against any missing or corrupt date rows
        if None in (min_prebd, max_plus8) or 0 in (min_prebd, max_plus8):
            continue
            
        # Parse start date string into datetime object
        s_start = str(int(min_prebd))
        start_date = datetime(int(s_start[0:4]), int(s_start[4:6]), int(s_start[6:8]))
        
        # Parse end date string into datetime object
        s_end = str(int(max_plus8))
        end_date = datetime(int(s_end[0:4]), int(s_end[4:6]), int(s_end[6:8]))
        
        # Calculate time difference
        delta = end_date - start_date
        
        # Update row
        cur.updateRow([min_prebd, max_plus8, abs(delta.days)])

print("Successfully calculated event_window_days.")